# ECON62020 - Group Replication Project

**Trinity Rose** (with two collaborators)

*This was a three-person group project. All members contributed jointly across data cleaning, the LASSO income-prediction stage, the TAXSIM tax-simulation pipeline, and the final IV regressions. The notebook below is the group's shared codebase.*

## Replication Paper

Leora Friedberg, Elliott Isaac, "Same-Sex Marriage Recognition and Taxes: New Evidence about the Impact of Household Taxation", The Review of Economics and Statistics (2024) 106 (1): 85–101.


# Summary

This project replicates key results from Friedberg and Isaac (2024), which examines how marriage-related tax incentives affect the likelihood of same-sex couples marrying. The key variable is the “marriage subsidy” (or penalty), defined as the difference between taxes paid when filing jointly versus separately.

Identification exploits variation in the timing of same-sex marriage recognition across U.S. states, alongside federal recognition following United States v. Windsor (2013). To isolate exogenous variation, the paper constructs a predicted marriage subsidy using LASSO-predicted earnings and uses this as an instrument for the observed subsidy.

This replication reconstructs Table A2 and reproduces the main results in Table 3. We find that higher marriage subsidies increase the probability of being married. While there are differences in levels due to tax simulation approximations, the direction and relative magnitude of the effects are consistent with the original findings.

# Data 

The analysis uses American Community Survey (ACS) data from 2003–2017, with the unit of observation being a same-sex couple classified as married or cohabiting. Couples are identified using partner links (sploc), sex, and detailed relationship codes, with adjustments for Census recoding.

The dataset includes demographic and income information for both partners. Income components include wages, business income, investment income, retirement income, Social Security income, and transfers, which are used to construct inputs for the NBER TAXSIM model.

The main outcome is an indicator for whether a couple is married. The key explanatory variable is the marriage subsidy, defined as the difference between joint and individual tax liabilities. Two measures are constructed: an observed subsidy based on reported income and a predicted subsidy based on LASSO-predicted earnings.

Control variables include age, education, differences between partners, number of children, race composition, and gender. The specification also includes state and year fixed effects, along with policy variables capturing marriage recognition and Medicaid expansion.

Due to data limitations, some tax inputs are approximated, which may lead to differences in levels relative to the original paper.

## Data Preparation

The data is constructed in two stages. First, a subset of variables is used to identify households containing same-sex couples using partner links and relationship codes. Second, a richer set of variables is extracted for these households only, and couples are formed by matching household heads to partners.

The sample is restricted to individuals aged 18–60 in U.S. states, excluding observations with allocated relationship values. Key variables are then constructed, including total earnings, within-couple earnings splits, demographic characteristics, and policy indicators.

In [ ]:
# Cell 1 - Import Libraries and Files
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import scipy.linalg as la
import warnings
warnings.filterwarnings('ignore')
from linearmodels.iv import IV2SLS
from statsmodels.tools import add_constant as sm_add_const
import io
import os
import subprocess

# Set project root relative to this notebook (no hardcoded personal path)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output"

In [ ]:
# Cell 3 - Import Cleaned Data

cleaned_data_path = OUTPUT_DIR / "cleaned_data.pkl"
lasso_input_path = OUTPUT_DIR / "acs_ssc_lasso_input_final.pkl"

df_final = pd.read_pickle(cleaned_data_path)

# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()

In [ ]:
# Cell 4 - Construct couple-level variables

# Income and earnings split calculation
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(df_f["total_earn"] > 0, df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"], np.nan)

# Construction of demographics and employment status variables
df_f["male"] = (df_f["sex_h"] == 1).astype(int)
df_f["female"] = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"] = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"] = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"] = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Racial matching and dependent children identification
df_f["same_race"] = ((df_f["race_h"] == df_f["race_s"]) & ((df_f["hispan_h"] > 0) == (df_f["hispan_s"] > 0))).astype(int)
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)

# Application of education years
df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])
df_f["edu_max"] = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"] = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]


# Output results comparison table
m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

In [ ]:
# Cell 5 - Define Helper Functions

def map_education_group(yrs):
    out = np.full(len(yrs), np.nan)
    out[yrs <  12] = 1
    out[yrs == 12] = 2
    out[(yrs > 12) & (yrs < 16)] = 3
    out[yrs >= 16] = 4
    return out

def map_age_group(age):
    out = np.full(len(age), np.nan)
    for upper in range(20, 105, 5):
        mask = (age <= upper) & (age > upper - 5)
        out[mask] = upper
    return out

def map_race(race, hispan):
    out = np.full(len(race), np.nan)
    out[(race == 2) & (hispan == 0)] = 1
    out[((race >= 4) & (race <= 6)) & (hispan == 0)] = 2
    out[((race == 3) | (race >= 7)) & (hispan == 0)] = 3
    out[(race == 1) & (hispan == 0)] = 4
    out[hispan > 0] = 5
    return out

def educd_to_yrs(s):
    s = pd.array(s, dtype="Int64")
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s == 14] = 1
    result[s == 15] = 2
    result[s == 16] = 3
    result[s == 17] = 4
    result[s == 22] = 5
    result[s == 23] = 6
    result[s == 25] = 7
    result[s == 26] = 8
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result


In [ ]:
# Cell 6 - Build Couple Level Variables

d = df_final.copy()
d["uhrswork_h"] = pd.to_numeric(d["uhrswork_h"], errors="coerce").fillna(0)
d["uhrswork_s"] = pd.to_numeric(d["uhrswork_s"], errors="coerce").fillna(0)
d["incearn_h"] = d["incearn_h"].clip(lower=0)
d["incearn_s"] = d["incearn_s"].clip(lower=0)

out = pd.DataFrame()
out["year"]         = d["year"]
out["serial"]       = d["serial"]
out["statefip"]     = d["statefip_h"]
out["sscouple_mar"] = d["married"].astype(bool)
out["sscouple_coh"] = (d["married"] == 0).astype(bool)
out["sscouple_all"] = True
out["r_male"]       = (d["sex_h"] == 1).astype(int)
out["age"]          = d["age_h"]
out["r_incearn"]    = d["incearn_h"]
r_yrs               = educd_to_yrs(d["educd_h"].fillna(-1).astype(int).values)
out["r_yrsedu"]     = r_yrs
out["r_edugroup"]   = map_education_group(r_yrs)
out["r_agegroup"]   = map_age_group(d["age_h"].values)
out["r_race"]       = map_race(d["race_h"].values, d["hispan_h"].values)
out["r_occbroad"]   = 0
out["r_degbroad"]   = 0
out["sp_male"]      = (d["sex_s"] == 1).astype(int)
out["sp_age"]       = d["age_s"]
out["sp_incearn"]   = d["incearn_s"]
sp_yrs              = educd_to_yrs(d["educd_s"].fillna(-1).astype(int).values)
out["sp_yrsedu"]    = sp_yrs
out["sp_edugroup"]  = map_education_group(sp_yrs)
out["sp_agegroup"]  = map_age_group(d["age_s"].values)
out["sp_race"]      = map_race(d["race_s"].values, d["hispan_s"].values)
out["sp_occbroad"]  = 0
out["sp_degbroad"]  = 0
out["c_children"]   = d["nchild_h"]
out["c_anychildren"]= (d["nchild_h"] > 0).astype(int)
out["c_racecomp"]   = (out["r_race"] == out["sp_race"]).astype(int)
out["c_agemax"]     = np.maximum(d["age_h"], d["age_s"])
out["c_agemin"]     = np.minimum(d["age_h"], d["age_s"])
out["c_agediff"]    = out["c_agemax"] - out["c_agemin"]
out["c_edumax"]     = np.maximum(r_yrs, sp_yrs)
out["c_edumin"]     = np.minimum(r_yrs, sp_yrs)
out["c_edudiff"]    = out["c_edumax"] - out["c_edumin"]
out["c_incearn"]    = d["incearn_h"] + d["incearn_s"]
total               = out["c_incearn"].replace(0, np.nan)
out["c_incearn_split"] = (np.maximum(d["incearn_h"], d["incearn_s"]) / total).fillna(0.5)
for i in range(1, 6):
    out[f"c_incearn{i}"]       = (out["c_incearn"] / 10_000) ** i
    out[f"c_incearn_split{i}"] =  out["c_incearn_split"] ** i
out["r_posincearn"]  = (d["incearn_h"] > 0).astype(int)
out["sp_posincearn"] = (d["incearn_s"] > 0).astype(int)
out["c_dualearner"]  = ((d["incearn_h"] > 0) & (d["incearn_s"] > 0)).astype(int)
out["c_singleearner"]= ((d["incearn_h"] > 0) ^ (d["incearn_s"] > 0)).astype(int)
out["c_noearner"]    = ((d["incearn_h"] == 0) & (d["incearn_s"] == 0)).astype(int)
out["migrate1"]      = d["migrate1_h"]


In [ ]:
# Cell 7 - Add Policy Indicators

ssm_start = {
    2: 2014, 4: 2014, 6: 2013, 8: 2014, 9: 2008, 10: 2013, 11: 2010,
    12: 2015, 15: 2013, 16: 2014, 17: 2014, 18: 2014, 19: 2009,
    23: 2012, 24: 2013, 25: 2004, 27: 2013, 30: 2014, 32: 2014,
    33: 2010, 34: 2013, 35: 2013, 36: 2011, 37: 2014, 40: 2014,
    41: 2014, 42: 2014, 44: 2013, 45: 2014, 49: 2014, 50: 2009,
    51: 2014, 53: 2012, 54: 2014, 55: 2014, 56: 2014
}
out["staterecog_policy"] = (
    out["year"] >= out["statefip"].map(ssm_start).fillna(2015)
).astype(int)
out["preW"]      = (out["year"] <= 2012).astype(int)
out["postWpreO"] = (out["year"].between(2013, 2014)).astype(int)
out["postO"]     = (out["year"] >= 2015).astype(int)

MEDICAID_EXP_STATES = {
    4, 6, 8, 9, 10, 11, 15, 17, 18, 19, 21, 23, 24, 25,
    26, 27, 32, 33, 34, 35, 36, 38, 41, 44, 50, 53, 55
}
out["medicaid_exp"] = (
    (out["year"] >= 2014) & (out["statefip"].isin(MEDICAID_EXP_STATES))
).astype(int)


In [ ]:
# Cell 8 - Save
print(f"Output shape: {out.shape}")
out.to_pickle(lasso_input_path)

# Table A2 

Table A2 reports summary statistics for the sample of same-sex couples used in the analysis, split between married and cohabiting couples. The table is designed to demonstrate that the constructed dataset closely matches the sample and descriptive statistics reported in the original paper.

In [ ]:
# Cell 1 - Print Table A2

print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Our Married':>18} {'Our Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male", m["male"], c["male"], "0.469 (0.499)", "0.506 (0.500)")
row("Female", m["female"], c["female"], "0.531 (0.499)", "0.494 (0.500)")
row("Same race", m["same_race"], c["same_race"], "0.793 (0.405)", "0.757 (0.429)")
row("Age older", m["age_old"], c["age_old"], "46.205 (9.633)","43.353 (10.902)")
row("Age younger", m["age_yng"], c["age_yng"], "41.252 (9.831)","37.858 (10.442)")
row("Age diff", m["age_diff"], c["age_diff"], "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)", m["edu_max"], c["edu_max"], "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)", m["edu_min"], c["edu_min"], "13.718 (3.044)","13.513 (2.604)")
row("Edu diff", m["edu_diff"], c["edu_diff"], "1.875 (2.286)", "1.850 (2.108)")
row("Any children", m["any_children"], c["any_children"], "0.314 (0.464)","0.162 (0.369)")
row("Both work", m["both_work"], c["both_work"], "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works", m["one_work"], c["one_work"], "0.195 (0.396)", "0.160 (0.367)")
row("Neither works", m["none_work"], c["none_work"], "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings", m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split", m.loc[m["total_earn"]>0,"earn_split"], c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")
print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

The resulting sample closely matches the original paper. We obtain 16,146 married and 21,761 cohabiting couples, compared to 16,098 and 21,136 in the paper. Demographic characteristics, labour supply measures, and income variables align almost exactly. For example, average earnings and earnings shares are nearly identical across both datasets.

Minor differences arise in variables such as the share of couples with children, likely due to small differences in variable construction. Overall, the close match provides strong validation of the data preparation process and supports the reliability of subsequent analysis.

# Table 3

To address endogeneity in observed earnings, we implement the paper’s two-step LASSO procedure. First, LASSO is used to predict labour force participation. Second, conditional on participation, LASSO predicts individual earnings using demographic characteristics. The model is trained on 2012 data and applied to other years.

Predicted earnings are aggregated to the couple level and used as inputs into the TAXSIM model (version 35) to simulate tax liabilities under joint and separate filing. The difference defines the predicted marriage subsidy, which depends only on exogenous characteristics.

We then estimate the causal effect of tax incentives using an instrumental variables approach, where the predicted subsidy instruments for the observed subsidy. This isolates variation in marriage incentives driven by the tax system rather than behavioural responses, allowing for a causal interpretation of the effect of financial incentives on marriage decisions.

## Two-Step LASSO

In [ ]:
# Cell 1 - Load Files

Predicted_output  = OUTPUT_DIR / "acs_ssc_predicted_second.pkl"

df_lasso = pd.read_pickle(lasso_input_path)
print(f"Loaded: {df_lasso.shape}  |  Years: {sorted(df_lasso['year'].unique())}")


In [ ]:
# Cell 2 - Check Columns

needed = [
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children", "statefip", "r_incearn", "sp_incearn"
]
missing = [c for c in needed if c not in df_lasso.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")
print("All columns present")

In [ ]:
# Cell 3 - Feature builder

def _dummies(s: pd.Series, prefix: str) -> pd.DataFrame:
    """One-hot encode a categorical series, dropping one base category."""
    return pd.get_dummies(s.astype("category"), prefix=prefix, drop_first=True, dtype=float)


def _add_interactions(
    out: dict,
    A: pd.DataFrame,
    B: pd.DataFrame,
    min_support: int = 1,
    same_source: bool = False
) -> None:
    """
    Add pairwise interactions between columns of A and B.
    If same_source=True, only keep upper triangle to avoid duplicates.
    min_support can be raised if memory becomes a problem.
    """
    a_cols = list(A.columns)
    b_cols = list(B.columns)

    for i, ca in enumerate(a_cols):
        start_j = i if same_source else 0
        for j in range(start_j, len(b_cols)):
            cb = b_cols[j]

            if same_source and ca == cb:
                # Keep squared terms separately where needed
                continue

            x = A[ca].to_numpy() * B[cb].to_numpy()
            if x.sum() >= min_support:
                out[f"{ca}__X__{cb}"] = x


def build_features(
    df: pd.DataFrame,
    prefix: str = "r",
    min_support: int = 10
) -> pd.DataFrame:
    req = [
        f"{prefix}_agegroup",
        f"{prefix}_edugroup",
        f"{prefix}_race",
        f"{prefix}_occbroad",
        f"{prefix}_degbroad",
        f"{prefix}_male",
        "statefip",
        "year",
        "c_children",
    ]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for prefix='{prefix}': {missing}")

    # Map Python names to STATA-style conceptual blocks
    age = _dummies(df[f"{prefix}_agegroup"], f"{prefix}_agegroup")
    edu = _dummies(df[f"{prefix}_edugroup"], f"{prefix}_edugroup")
    race = _dummies(df[f"{prefix}_race"], f"{prefix}_race")
    occ = _dummies(df[f"{prefix}_occbroad"], f"{prefix}_occbroad")
    deg = _dummies(df[f"{prefix}_degbroad"], f"{prefix}_degbroad")
    state = _dummies(df["statefip"], "statefip")
    year = _dummies(df["year"], "year")

    sex = pd.DataFrame({f"{prefix}_male": df[f"{prefix}_male"].astype(float).to_numpy()}, index=df.index)
    children = pd.DataFrame({"c_children": df["c_children"].astype(float).to_numpy()}, index=df.index)

    out = {}

    # Main effects
    for block in [age, edu, race, occ, deg, state, year, sex]:
        for c in block.columns:
            out[c] = block[c].to_numpy()

    # children main + square (STATA: c.c_children##c.c_children)
    out["c_children"] = df["c_children"].astype(float).to_numpy()
    out["c_children_sq"] = np.square(df["c_children"].astype(float).to_numpy())

    # age block
    _add_interactions(out, age, age, min_support=min_support, same_source=True)
    _add_interactions(out, age, edu, min_support=min_support)
    _add_interactions(out, age, race, min_support=min_support)
    _add_interactions(out, age, sex, min_support=min_support)
    _add_interactions(out, age, occ, min_support=min_support)
    _add_interactions(out, age, deg, min_support=min_support)
    _add_interactions(out, age, state, min_support=min_support)
    _add_interactions(out, age, year, min_support=min_support)
    _add_interactions(out, age, children, min_support=min_support)

    # education block
    _add_interactions(out, edu, edu, min_support=min_support, same_source=True)
    _add_interactions(out, edu, race, min_support=min_support)
    _add_interactions(out, edu, sex, min_support=min_support)
    _add_interactions(out, edu, occ, min_support=min_support)
    _add_interactions(out, edu, deg, min_support=min_support)
    _add_interactions(out, edu, state, min_support=min_support)
    _add_interactions(out, edu, year, min_support=min_support)
    _add_interactions(out, edu, children, min_support=min_support)

    # race block
    _add_interactions(out, race, sex, min_support=min_support)
    _add_interactions(out, race, occ, min_support=min_support)
    _add_interactions(out, race, deg, min_support=min_support)
    _add_interactions(out, race, state, min_support=min_support)
    _add_interactions(out, race, year, min_support=min_support)
    _add_interactions(out, race, children, min_support=min_support)

    # sex block
    _add_interactions(out, sex, occ, min_support=min_support)
    _add_interactions(out, sex, deg, min_support=min_support)
    _add_interactions(out, sex, state, min_support=min_support)
    _add_interactions(out, sex, year, min_support=min_support)
    _add_interactions(out, sex, children, min_support=min_support)

    # occupation block
    _add_interactions(out, occ, deg, min_support=min_support)
    _add_interactions(out, occ, state, min_support=min_support)
    _add_interactions(out, occ, year, min_support=min_support)
    _add_interactions(out, occ, children, min_support=min_support)

    # degree block
    _add_interactions(out, deg, state, min_support=min_support)
    _add_interactions(out, deg, year, min_support=min_support)
    _add_interactions(out, deg, children, min_support=min_support)

    # state / year / children blocks
    _add_interactions(out, state, year, min_support=min_support)
    _add_interactions(out, state, children, min_support=min_support)
    _add_interactions(out, year, children, min_support=min_support)

    X = pd.DataFrame(out, index=df.index).astype(float)

    # Drop constant columns if any survive
    nunique = X.nunique(dropna=False)
    X = X.loc[:, nunique > 1]

    return X


def predict_income_for_role(
    df: pd.DataFrame,
    train_mask: pd.Series,
    prefix: str = "r",
    cv: int = 10,
    max_iter: int = 20_000,
    min_support: int = 10,
    positive_threshold: float | None = None,
):
    earn_col = f"{prefix}_incearn"

    X_full = build_features(df, prefix=prefix, min_support=min_support)
    scaler = StandardScaler(with_mean=False)  
    X_sc = scaler.fit_transform(X_full)

    # Step 1: positive earnings
    y1 = (df[earn_col] > 0).astype(float).to_numpy()
    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_sc[train_mask.to_numpy()], y1[train_mask.to_numpy()])
    score1 = m1.predict(X_sc)

    if positive_threshold is None:
        target_share = y1.mean()
        positive_threshold = np.percentile(score1, (1 - target_share) * 100)

    hat_pos = (score1 >= positive_threshold).astype(float)

    # Step 2: earnings conditional on positive earnings
    pos_train = train_mask.to_numpy() & (df[earn_col].to_numpy() > 0)
    y2 = df.loc[pos_train, earn_col].to_numpy()

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X_sc[pos_train], y2)
    pred_level = np.maximum(m2.predict(X_sc), 0.0)

    out = df.copy()
    out[f"hat_pos_{prefix}"] = hat_pos
    out[f"hat_incearn_{prefix}"] = np.where(hat_pos == 1, pred_level, 0.0)

    info = {
        "X_shape": X_full.shape,
        "n_selected_step1": int(np.sum(m1.coef_ != 0)),
        "n_selected_step2": int(np.sum(m2.coef_ != 0)),
        "threshold": float(positive_threshold),
        "achieved_positive_share": float(hat_pos.mean()),
        "reported_positive_share": float(y1.mean()),
    }
    return out, X_full, m1, m2, info

In [ ]:
train_mask = (df_lasso["year"] == 2012)

# Respondent
df2, X_r, m1_r, m2_r, info_r = predict_income_for_role(
    df_lasso,
    train_mask=train_mask,
    prefix="r",
    cv=10,
    max_iter=20000,
    min_support=10,
    positive_threshold=None,   # better than hard-coding 0.78 unless you find the package rule
)

print("Respondent:", info_r)

# Spouse
df2, X_sp, m1_sp, m2_sp, info_sp = predict_income_for_role(
    df2,
    train_mask=train_mask,
    prefix="sp",
    cv=10,
    max_iter=20000,
    min_support=10,
    positive_threshold=None,
)

print("Spouse:", info_sp)

In [ ]:
# Cell 4 - Couple-level aggregates
df2["hat_c_incearn"] = df2["hat_incearn_r"] + df2["hat_incearn_sp"]

total = df2["hat_c_incearn"].replace(0, np.nan)
df2["hat_c_split"] = (
    np.maximum(df2["hat_incearn_r"], df2["hat_incearn_sp"]) / total
).fillna(0.5)

for i in range(1, 6):
    df2[f"hat_c_incearn{i}"] = (df2["hat_c_incearn"] / 10_000) ** i
    df2[f"hat_c_split{i}"] = df2["hat_c_split"] ** i

In [ ]:
# Cell 5 - Diagnostics vs Table 2
print("=" * 60)
print("DIAGNOSTICS vs. Table 2")
print("=" * 60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df2['r_incearn'] > 0).mean():.3f}  "
      f"predicted={df2['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df2['sp_incearn'] > 0).mean():.3f}  "
      f"predicted={df2['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df2["sscouple_mar"]),
                    ("Cohabiting", df2["sscouple_coh"])]:
    sub = df2.loc[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df2["sscouple_mar"]),
                    ("Cohabiting", df2["sscouple_coh"])]:
    sub = df2.loc[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos, 'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df2[f"{pfx}_incearn"] > 0) & (df2[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(
        df2.loc[both, f"{pfx}_incearn"],
        df2.loc[both, f"hat_incearn_{pfx}"]
    )[0, 1]
    print(f"  {label}: r={r:.3f}")

In [ ]:
# Cell 6 - Save 
df2.to_pickle(Predicted_output)
print(f"Saved to {Predicted_output}")

## TAXSIM35

In [ ]:
# Cell 1 - TAXSIM35 Paths
taxsim_output = OUTPUT_DIR / "acs_ssc_taxsim_results_second.pkl"
TAXSIM_EXE = os.path.expanduser("~/taxsim35")

if not os.path.exists(TAXSIM_EXE):
    raise FileNotFoundError(
        f"Binary not found at {TAXSIM_EXE}.\n"
        "Run: curl -o ~/taxsim35 "
        "https://taxsim.nber.org/stata/taxsim35/taxsim35-osx.exe "
        "&& chmod +x ~/taxsim35"
    )

df_pred = pd.read_pickle(Predicted_output)
print(f"Loaded: {df_pred.shape}")
print(f"Years:  {sorted(df_pred['year'].unique())}")
print(f"Married:    {(df_pred['sscouple_mar']==True).sum():,}")
print(f"Cohabiting: {(df_pred['sscouple_mar']==False).sum():,}")

In [ ]:
# Cell 2 - State code conversion
# TAXSIM uses SOI codes not FIPS.

FIPS_TO_SOI = {
    1:  1,  2:  2,  4:  3,  5:  4,  6:  5,  8:  6,  9:  7,
    10: 8,  11: 9,  12: 10, 13: 11, 15: 12, 16: 13, 17: 14,
    18: 15, 19: 16, 20: 17, 21: 18, 22: 19, 23: 20, 24: 21,
    25: 22, 26: 23, 27: 24, 28: 25, 29: 26, 30: 27, 31: 28,
    32: 29, 33: 30, 34: 31, 35: 32, 36: 33, 37: 34, 38: 35,
    39: 36, 40: 37, 41: 38, 42: 39, 44: 40, 45: 41, 46: 42,
    47: 43, 48: 44, 49: 45, 50: 46, 51: 47, 53: 48, 54: 49,
    55: 50, 56: 51,
}

unmapped = set(df_pred["statefip"].unique()) - set(FIPS_TO_SOI.keys())
print(f"Unmapped FIPS: {unmapped if unmapped else 'None'}")

In [ ]:
# Cell 3 - TAXSIM35 chunked runner 

def run_taxsim35(df_taxsim: pd.DataFrame, chunksize: int = 1000,
                 label: str = "") -> pd.DataFrame:
    chunks_out = []
    n = len(df_taxsim)
    n_chunks = (n + chunksize - 1) // chunksize
    for i in range(n_chunks):
        chunk = df_taxsim.iloc[i * chunksize : (i + 1) * chunksize].copy()
        chunk["taxsimid"] = np.arange(1, len(chunk) + 1)
        result = subprocess.run(
            [TAXSIM_EXE],
            input=chunk.to_csv(index=False),
            capture_output=True, text=True, timeout=300,
        )
        stdout = result.stdout.strip()
        if not stdout or not stdout.startswith("taxsimid"):
            raise RuntimeError(
                f"{label} chunk {i+1}/{n_chunks} failed.\n"
                f"stderr: {result.stderr[:1000]}\n"
                f"stdout: {stdout[:500]}"
            )
        chunks_out.append(pd.read_csv(io.StringIO(result.stdout)))
        if (i + 1) % 10 == 0 or i == n_chunks - 1:
            print(f"  {label}: {i+1}/{n_chunks} chunks "
                  f"({min((i+1)*chunksize, n):,}/{n:,} rows)")
    return pd.concat(chunks_out, ignore_index=True)


# ── Input builders ────────────────────────────────────────────────
# Per paper p.89: 'no other tax expenditures' means depx=0 in ALL runs.
# Children are controlled for separately in the regression.
# Both single filers use mstat=1 (plain single, no HoH).

def make_single(df_pred, wage_col, age_col):
    """Single filer — earned income only, passed through pwages, no children, mstat=1."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df_pred) + 1)
    t["year"]   = df_pred["year"].values
    t["state"]  = df_pred["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 1
    t["page"]   = df_pred[age_col].values
    t["sage"]   = 0
    t["pwages"] = df_pred[wage_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = 0
    t["depx"]   = 0
    return t


def make_joint(df_pred, wage_r_col, wage_sp_col, age_r_col, age_sp_col):
    """Joint filer — earned income only, passed through pwages/swages, no children, mstat=2."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df_pred) + 1)
    t["year"]   = df_pred["year"].values
    t["state"]  = df_pred["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 2
    t["page"]   = df_pred[age_r_col].values
    t["sage"]   = df_pred[age_sp_col].values
    t["pwages"] = df_pred[wage_r_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = df_pred[wage_sp_col].clip(lower=0).fillna(0).round().astype(int)
    t["depx"]   = 0
    return t
print("Helpers defined.")


In [ ]:
# Cell 4 - Run A: Predicted Income
# uses hat_incearn_r / hat_incearn_sp from LASSO

print("Run A1: respondent as single (predicted)...")
a1_out = run_taxsim35(
    make_single(df_pred, "hat_incearn_r", "age"),
    label="A1")

print("Run A2: spouse as single (predicted)...")
a2_out = run_taxsim35(
    make_single(df_pred, "hat_incearn_sp", "sp_age"),
    label="A2")

print("Run A3: couple jointly (predicted)...")
a3_out = run_taxsim35(
    make_joint(df_pred, "hat_incearn_r", "hat_incearn_sp", "age", "sp_age"),
    label="A3")

assert len(a1_out) == len(a2_out) == len(a3_out) == len(df_pred), "Row mismatch!"
print(f"\nA runs done.")
print(f"  A1 mean fiitax: {a1_out['fiitax'].mean():,.2f}")
print(f"  A2 mean fiitax: {a2_out['fiitax'].mean():,.2f}")
print(f"  A3 mean fiitax: {a3_out['fiitax'].mean():,.2f}")
print(f"  Raw fed subsidy (all): {(a1_out['fiitax']+a2_out['fiitax']-a3_out['fiitax']).mean():,.2f}")

In [ ]:
# Cell 5 - Run B: Reported Income
# uses r_incearn / sp_incearn from ACS

print("Run B1: respondent as single (reported)...")
b1_out = run_taxsim35(
    make_single(df_pred, "r_incearn", "age"),
    label="B1")

print("Run B2: spouse as single (reported)...")
b2_out = run_taxsim35(
    make_single(df_pred, "sp_incearn", "sp_age"),
    label="B2")

print("Run B3: couple jointly (reported)...")
b3_out = run_taxsim35(
    make_joint(df_pred, "r_incearn", "sp_incearn", "age", "sp_age"),
    label="B3")

assert len(b1_out) == len(b2_out) == len(b3_out) == len(df_pred), "Row mismatch!"
print(f"\nB runs done.")
print(f"  Raw fed subsidy (all): {(b1_out['fiitax']+b2_out['fiitax']-b3_out['fiitax']).mean():,.2f}")

In [ ]:
# Cell 6 - Compute marriage subsidies
# subsidy = (T_i + T_j) - T_c
# positive = subsidy (married pay less); negative = penalty.

df_out = df_pred.copy()
m = df_out["sscouple_mar"] == True
c = df_out["sscouple_mar"] == False

# Predicted income subsidies (instrument)
df_out["fed_sub_pred"]   = (a1_out["fiitax"].values + a2_out["fiitax"].values
                            - a3_out["fiitax"].values)
df_out["state_sub_pred"] = (a1_out["siitax"].values + a2_out["siitax"].values
                            - a3_out["siitax"].values)
df_out["total_sub_pred"] = df_out["fed_sub_pred"] + df_out["state_sub_pred"]

# Reported income subsidies (observed)
df_out["fed_sub_obs"]    = (b1_out["fiitax"].values + b2_out["fiitax"].values
                            - b3_out["fiitax"].values)
df_out["state_sub_obs"]  = (b1_out["siitax"].values + b2_out["siitax"].values
                            - b3_out["siitax"].values)
df_out["total_sub_obs"]  = df_out["fed_sub_obs"] + df_out["state_sub_obs"]

# Joint filing totals (for other analyses)
df_out["fiitax_joint_pred"] = a3_out["fiitax"].values
df_out["siitax_joint_pred"] = a3_out["siitax"].values
df_out["fica_joint_pred"]   = a3_out["fica"].values
df_out["fiitax_joint_obs"]  = b3_out["fiitax"].values
df_out["siitax_joint_obs"]  = b3_out["siitax"].values
df_out["fica_joint_obs"]    = b3_out["fica"].values

print("Subsidies computed.")
print(f"\nQuick check vs paper:")
print(f"  Fed sub pred  married: {df_out.loc[m,'fed_sub_pred'].mean():>10,.2f}  (paper: 122.41)")
print(f"  Fed sub pred  cohab:   {df_out.loc[c,'fed_sub_pred'].mean():>10,.2f}  (paper: 266.89)")
print(f"  Fed sub obs   married: {df_out.loc[m,'fed_sub_obs'].mean():>10,.2f}  (paper: 395.05)")
print(f"  Fed sub obs   cohab:   {df_out.loc[c,'fed_sub_obs'].mean():>10,.2f}  (paper: 231.80)")
print(f"  State sub pred married:{df_out.loc[m,'state_sub_pred'].mean():>10,.2f}  (paper: -54.21)")
print(f"  State sub pred cohab:  {df_out.loc[c,'state_sub_pred'].mean():>10,.2f}  (paper: -10.06)")

In [ ]:
# Cell 7 - Replicate Table 2

def ms(series, mask, fmt=".2f"):
    v = series[mask]
    return f"{v.mean():{fmt}}  ({v.std():{fmt}})"

rep_total    = df_out["r_incearn"] + df_out["sp_incearn"]
pos_rep_m    = m & (rep_total > 0)
pos_rep_c    = c & (rep_total > 0)
rep_split_m  = (df_out.loc[pos_rep_m, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_m])
rep_split_c  = (df_out.loc[pos_rep_c, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_c])
pos_pred_m   = m & (df_out["hat_c_incearn"] > 0)
pos_pred_c   = c & (df_out["hat_c_incearn"] > 0)
pred_split_m = df_out.loc[pos_pred_m, "hat_c_split"]
pred_split_c = df_out.loc[pos_pred_c, "hat_c_split"]

rows = [
    ("Positive earnings (reported)",
     ms((rep_total>0).astype(float), m, ".3f"),   "0.935 (0.246)",
     ms((rep_total>0).astype(float), c, ".3f"),   "0.935 (0.247)"),
    ("Positive earnings (predicted)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), m, ".3f"),
     "0.963 (0.188)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), c, ".3f"),
     "0.969 (0.172)"),
    ("Reported earnings",
     ms(rep_total, m, ",.2f"),   "125,286.76 (119,779.91)",
     ms(rep_total, c, ",.2f"),   "105,188.00 (105,191.59)"),
    ("Predicted earnings",
     ms(df_out["hat_c_incearn"], m, ",.2f"),   "110,729.40 (57,936.40)",
     ms(df_out["hat_c_incearn"], c, ",.2f"),   "102,952.54 (54,275.74)"),
    ("Reported earnings split",
     f"{rep_split_m.mean():.3f}  ({rep_split_m.std():.3f})",   "0.745 (0.200)",
     f"{rep_split_c.mean():.3f}  ({rep_split_c.std():.3f})",   "0.723 (0.174)"),
    ("Predicted earnings split",
     f"{pred_split_m.mean():.3f}  ({pred_split_m.std():.3f})",   "0.648 (0.197)",
     f"{pred_split_c.mean():.3f}  ({pred_split_c.std():.3f})",   "0.641 (0.181)"),
    ("Fed+state subsidy (reported)",
     ms(df_out["total_sub_obs"], m, ",.2f"),   "442.45 (5,116.62)",
     ms(df_out["total_sub_obs"], c, ",.2f"),   "263.79 (3,247.05)"),
    ("Fed+state subsidy (predicted)",
     ms(df_out["total_sub_pred"], m, ",.2f"),   "68.19 (2,218.99)",
     ms(df_out["total_sub_pred"], c, ",.2f"),   "256.82 (1,623.22)"),
    ("Fed subsidy (reported)",
     ms(df_out["fed_sub_obs"], m, ",.2f"),   "395.05 (4,563.36)",
     ms(df_out["fed_sub_obs"], c, ",.2f"),   "231.80 (3,055.28)"),
    ("Fed subsidy (predicted)",
     ms(df_out["fed_sub_pred"], m, ",.2f"),   "122.41 (1,896.07)",
     ms(df_out["fed_sub_pred"], c, ",.2f"),   "266.89 (1,427.33)"),
    ("State subsidy (reported)",
     ms(df_out["state_sub_obs"], m, ",.2f"),   "47.41 (974.14)",
     ms(df_out["state_sub_obs"], c, ",.2f"),   "31.99 (584.34)"),
    ("State subsidy (predicted)",
     ms(df_out["state_sub_pred"], m, ",.2f"),   "-54.21 (487.06)",
     ms(df_out["state_sub_pred"], c, ",.2f"),   "-10.06 (332.98)"),
]

w = 42
print(f"{'TABLE 2':^126}")
print(f"{'':>{w}} {'Replication (Married)':>28} {'Paper (Married)':>24} "
      f"{'Replication (Cohab)':>28} {'Paper (Cohab)':>22}")
print("─" * 126)
for label, ym, pm, yc, pc in rows:
    print(f"{label:<{w}} {ym:>28} {pm:>24} {yc:>28} {pc:>22}")
print("─" * 126)
print(f"{'Observations':<{w}} {m.sum():>28,} {'16,098':>24} "
      f"{c.sum():>28,} {'21,136':>22}")

In [ ]:
# Cell 8 - Subsidy by Year 

print("=== Mean predicted total subsidy by year ===")
piv = df_out.groupby(["year","sscouple_mar"])["total_sub_pred"].mean().unstack()
piv.columns = ["Cohabiting", "Married"]
piv["Married - Cohabiting"] = piv["Married"] - piv["Cohabiting"]
print(piv.round(1).to_string())

print("\n=== Mean observed total subsidy by year ===")
piv2 = df_out.groupby(["year","sscouple_mar"])["total_sub_obs"].mean().unstack()
piv2.columns = ["Cohabiting", "Married"]
piv2["Married - Cohabiting"] = piv2["Married"] - piv2["Cohabiting"]
print(piv2.round(1).to_string())

In [ ]:
# Cell 9 - Save 
new_cols = [
    "fed_sub_pred",  "state_sub_pred", "total_sub_pred",
    "fed_sub_obs",   "state_sub_obs",  "total_sub_obs",
    "fiitax_joint_pred", "siitax_joint_pred", "fica_joint_pred",
    "fiitax_joint_obs",  "siitax_joint_obs",  "fica_joint_obs",
]
df_out.to_pickle(taxsim_output)
print(f"Saved → {taxsim_output}")
print(f"Shape: {df_out.shape}")
print(f"New columns: {new_cols}")

## Regressions

In [ ]:
# Cell 1 - Load Data
# IN_PKL = "/Users/trinityrose/Desktop/acs_ssc_msub.pkl"

df_reg = pd.read_pickle(taxsim_output)
print(f"Loaded: {df_reg.shape}")
print(f"Married share: {df_reg['sscouple_mar'].mean():.3f}")

# legal_marriage not yet in pickle — build from legalisation dates
ssm_legalise = {
    6: 2008, 9: 2008, 25: 2004, 11: 2010, 15: 2013,
    19: 2009, 23: 2012, 24: 2013, 27: 2013, 33: 2010,
    34: 2013, 36: 2011, 44: 2013, 50: 2009, 53: 2012,
}
df_reg["legal_marriage"] = (
    df_reg["year"] >= df_reg["statefip"].map(ssm_legalise).fillna(2015)
).astype(float)

In [ ]:
# Cell 2 — Prepare variables
df_reg["married"]        = df_reg["sscouple_mar"].astype(float)
df_reg["msub_obs_k"]     = df_reg["total_sub_obs"]  / 1000
df_reg["msub_hat_k"]     = df_reg["total_sub_pred"] / 1000

df_reg["legal_marriage"] = df_reg["legal_marriage"].astype(float)

df_reg["male"]         = df_reg["r_male"].astype(float)
df_reg["any_children"] = df_reg["c_anychildren"].astype(float)
df_reg["n_children"]   = df_reg["c_children"].astype(float)
df_reg["age_older"]    = df_reg["c_agemax"].astype(float)
df_reg["age_diff"]     = df_reg["c_agediff"].astype(float)
df_reg["edu_max"]      = df_reg["c_edumax"].astype(float)
df_reg["edu_diff"]     = df_reg["c_edudiff"].astype(float)
df_reg["same_race"]    = df_reg["c_racecomp"].astype(float)
df_reg["medicaid_exp"] = df_reg["medicaid_exp"].astype(float)
df_reg["earn_split_obs"] = df_reg["c_incearn_split"].astype(float)
df_reg["earn_split_hat"] = df_reg["hat_c_split"].astype(float)

# Check c_incearn 
assert "c_incearn" in df_reg.columns, "c_incearn missing from df_reg — check lasso_input save"

# Predicted earnings polynomials (powers 1–5); hat_split not needed
for i in range(1, 6):
    df_reg[f"hat_earn{i}"] = df_reg[f"hat_c_incearn{i}"].astype(float)

df_reg["year_fe"]  = df_reg["year"].astype(str)
df_reg["state_fe"] = df_reg["statefip"].astype(str)

key_cols = ["married", "msub_obs_k", "msub_hat_k", "legal_marriage",
            "male", "any_children", "n_children", "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race", "medicaid_exp",
            "earn_split_obs", "earn_split_hat", "c_incearn"]
df_reg = df_reg.dropna(subset=key_cols).copy()
print(f"Analysis sample: {len(df_reg):,}")
print(f"Married share:   {df_reg['married'].mean():.3f}  (paper: 0.432)")

In [ ]:
# Cell 3 — Helpers
def make_dummies(df, col):
    return pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=float)


def clean_design_matrix(X: pd.DataFrame) -> pd.DataFrame:
    # drop constant columns
    X = X.loc[:, X.nunique(dropna=False) > 1]

    # drop exact duplicate columns
    X = X.T.drop_duplicates().T

    return X

def run_spec(df, extra_controls=None, iv=False):
    extra_controls = extra_controls or []
    base = ["legal_marriage", "medicaid_exp", "male",
            "any_children", "n_children",
            "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race"]

    year_dums = make_dummies(df, "year_fe")
    state_dums = make_dummies(df, "state_fe")

    X_cols = base + extra_controls
    X = pd.concat([df[X_cols], year_dums, state_dums], axis=1).astype(float)
    X = sm_add_const(X)

    # add observed subsidy for OLS spec
    if not iv:
        X.insert(1, "msub_obs_k", df["msub_obs_k"].values)

    # important: remove constant / duplicate columns
    X = clean_design_matrix(X)

    y = df["married"]

    if not iv:
        res = IV2SLS(y, X, None, None).fit(cov_type="robust")
    else:
        endog = df[["msub_obs_k"]]
        instr = df[["msub_hat_k"]]
        res = IV2SLS(y, X, endog, instr).fit(cov_type="robust")

    return res

def first_stage_info(res):
    try:
        fs    = res.first_stage
        indiv = fs.individual["msub_obs_k"]
        c     = float(indiv.params["msub_hat_k"])
        se    = float(indiv.std_errors["msub_hat_k"])
        p     = float(indiv.pvalues["msub_hat_k"])
        f     = float(fs.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

print("Helpers defined.")

In [ ]:
# Cell 4

# Cols 3-4: polynomial + split, NO control function
poly_cols_obs = [f"obs_earn{i}" for i in range(1, 6)] + ["earn_split_obs"]
poly_cols_hat = [f"hat_earn{i}" for i in range(1, 6)] + ["earn_split_hat"]

# Cols 5-6: polynomial + split + control function
year_dums_cf  = pd.get_dummies(df_reg["year_fe"],  prefix="year_fe",  drop_first=True, dtype=float)
state_dums_cf = pd.get_dummies(df_reg["state_fe"], prefix="state_fe", drop_first=True, dtype=float)

cf_rhs = pd.concat(
    [df_reg[poly_cols_hat], year_dums_cf, state_dums_cf], axis=1
).astype(float)
cf_rhs = sm.add_constant(cf_rhs)

df_reg["cf_resid"] = sm.OLS(df_reg["c_incearn"].values, cf_rhs.values).fit().resid

cf_cols_ols = poly_cols_obs + ["cf_resid"]   # OLS: observed poly + residual
cf_cols_iv  = poly_cols_hat + ["cf_resid"]   # IV:  predicted poly + residual

print("Col 1: OLS, no income controls...")
col1 = run_spec(df_reg, extra_controls=[],             iv=False)
print("Col 2: IV,  no income controls...")
col2 = run_spec(df_reg, extra_controls=[],             iv=True)
print("Col 3: OLS, polynomial + split...")
col3 = run_spec(df_reg, extra_controls=poly_cols_obs,  iv=False)
print("Col 4: IV,  polynomial + split...")
col4 = run_spec(df_reg, extra_controls=poly_cols_hat,  iv=True)
print("Col 5: OLS, polynomial + split + control function...")
col5 = run_spec(df_reg, extra_controls=cf_cols_ols,    iv=False)
print("Col 6: IV,  polynomial + split + control function...")
col6 = run_spec(df_reg, extra_controls=cf_cols_iv,     iv=True)

print("All columns estimated.")

In [ ]:
# Cell 5 — Display table
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "   "

def get_coef_se_p(res, name):
    if name not in res.params.index:
        return None, None, None
    return res.params[name], res.std_errors[name], res.pvalues[name]

def first_stage_info(res):
    try:
        indiv = res.first_stage.individual["msub_obs_k"]
        c  = float(indiv.params["msub_hat_k"])
        se = float(indiv.std_errors["msub_hat_k"])
        p  = float(indiv.pvalues["msub_hat_k"])
        f  = float(res.first_stage.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

results  = [col1, col2, col3, col4, col5, col6]
is_iv    = [False, True, False, True, False, True]
mean_dv  = df_reg["married"].mean()

sep = "-" * 105

print("=" * 105)
print("TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)")
print("=" * 105)
hdr = f"{'':42}" + "".join(f"{'OLS' if not v else 'IV':>11}" for v in is_iv)
print(hdr)
print(sep)

def print_row(label, varname, paper_vals):
    # Coefficient row
    line = f"{label:42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if c is not None:
            line += f"  {c:7.3f}{stars(p)}"
        else:
            line += f"  {'—':>10}"
    print(line)
    # SE row
    line = f"{'':42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if se is not None:
            line += f"  ({se:6.3f})  "
        else:
            line += f"  {'':>10}"
    print(line)
    # Paper row
    print(f"  {'(paper)':40}" + "".join(f"  {v:>10}" for v in paper_vals))
    print()

print_row("Marriage subsidy ($1,000s)", "msub_obs_k",
          ["0.005***", "0.008***", "0.004***", "0.009*  ", "0.005***", "0.014***"])
print_row("Legal marriage", "legal_marriage",
          ["0.066***", "0.066***", "0.066***", "0.066***", "0.116***", "0.116***"])
print_row("State Medicaid expansion", "medicaid_exp",
          ["0.010   ", "0.010   ", "0.009   ", "0.010   ", "0.035***", "0.034***"])

print(sep)

# First stage
print(f"{'First stage coef':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  {c:7.3f}{stars(p)}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()

print(f"{'':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  ({se:6.3f})  ", end="")
    else:
        print(f"  {'':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'0.463***':>10}  {'—':>10}  {'0.408***':>10}  {'—':>10}  {'0.420***':>10}")
print()

print(f"{'First stage F-stat':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        _, _, _, f = first_stage_info(res)
        print(f"  {f:>10.1f}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'[474.7]':>10}  {'—':>10}  {'[221.0]':>10}  {'—':>10}  {'[261.3]':>10}")

print(sep)
print(f"{'Mean dep var':42}" + "".join(f"  {mean_dv:>10.3f}" for _ in results))
print(f"{'N':42}" + "".join(f"  {int(res.nobs):>10,}" for res in results))
print(f"  {'(paper N)':40}" + "".join(f"  {'37,234':>10}" for _ in results))
print("=" * 105)

In [ ]:
# Cell 6 — Calculate Elasticity

c6, _, _ = get_coef_se_p(col6, "msub_obs_k")
mean_msub = df_reg["msub_obs_k"].mean()
elasticity = c6 * (mean_msub / mean_dv)
print(f"IV coef (col 6):     {c6:.4f}")
print(f"Mean subsidy ($k):   {mean_msub:.3f}")
print(f"Mean married:        {mean_dv:.3f}")
print(f"Implied elasticity:  {elasticity:.4f}  (paper: 0.011)")

# Understanding and Explanation

Table 3 estimates a linear probability model where the outcome is an indicator for being married (vs. cohabiting). The key regressor is the marriage subsidy in thousands of dollars. OLS is problematic because couples may adjust labour supply in response to marriage or taxes (simultaneity bias) and because reported income is measured with error. The IV strategy addresses both issues: predicted earnings, based only on pre-treatment 2012 characteristics, are used to construct a predicted subsidy that is orthogonal to post-treatment labour supply decisions. The control function specification (columns 5–6) adds a fifth-order polynomial in predicted earnings and predicted earnings split to ensure identification is driven by tax law variation rather than any spurious correlation between earnings levels and marriage propensity.

# Shortcomings

The main limitation is the first stage LASSO. Our first-stage F-statistics range from 15–32, compared to 15–60 in the paper. This stems from two sources: our LASSO feature set omits college major codes (a strong earnings predictor), and our threshold calibration achieves employment shares of 0.957/0.923 versus the paper's 0.963/0.969, compressing predicted earnings and weakening the instrument. As a result, our IV estimates are less reliable than the paper's — our OLS results are the stronger point of comparison.

Our sample contains 37,907 couples versus the paper's 37,234, likely due to minor differences in relationship-status cleaning rules, though this does not materially affect results.

# Opportunities for Extensions

**Heterogeneous effects by earnings quantile using quantile IV regression.** The paper reports that marriage elasticities decline with household income (Figure 4), but uses a parametric polynomial interaction. An extension would estimate quantile IV regressions to obtain distributional treatment effects non-parametrically, testing whether the marriage response is concentrated among households at specific points of the earnings distribution rather than smoothly declining.

**Impact of the 2018 Tax Cuts and Jobs Act on different-sex couples.** The paper simulates TCJA effects for same-sex couples (Figure 5). The same machinery — LASSO income prediction + TAXSIM + marriage subsidy calculation — could be applied to different-sex cohabiting couples using ACS 2018–2022 data. This would test whether the TCJA-induced increase in marriage subsidies for high earners actually increased marriage rates among cohabiting heterosexual couples, providing a direct test of the paper's out-of-sample prediction with a much larger sample.